# Module 14 — Residual Connections

Module 13 fixed activation *scale* drifting across layers. There's a
second, separate problem with deep stacks: during backpropagation, the
gradient has to flow back through every single layer's Jacobian in turn.
If each layer shrinks the gradient even slightly, the shrinkage
**compounds multiplicatively** — 30 layers each shrinking the gradient by
half means the gradient reaching the first layer is `0.5^30`, indistinguishable
from zero. This is the **vanishing gradient problem**, and it means early
layers in a deep network barely learn anything at all.

**Residual (skip) connections** fix this with one of the simplest ideas in
deep learning: instead of `output = sublayer(x)`, compute
`output = x + sublayer(x)`. That `+ x` term means the gradient always has
one path straight through — of derivative exactly 1 — no matter how small
the sublayer's own gradient contribution is.

## 1. Vanishing gradients in a deep stack, no residuals

30 layers, each a small linear transform + `tanh`, with a weight scale
deliberately chosen so each layer shrinks its signal a bit — a completely
plausible real-world initialization, not a contrived worst case.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

d_model = 16
n_layers = 30
layers = [nn.Linear(d_model, d_model, bias=False) for _ in range(n_layers)]
for layer in layers:
    nn.init.normal_(layer.weight, mean=0.0, std=0.5 / (d_model ** 0.5))

x = torch.randn(1, d_model, requires_grad=True)

h = x
for layer in layers:
    h = torch.tanh(layer(h))
loss = h.sum()
loss.backward()

print(f"Final layer activation std (forward signal): {h.std().item():.2e}")
print(f"Gradient norm reaching the INPUT (backward signal): {x.grad.norm().item():.2e}")
print("\nBoth the forward signal and the gradient have vanished to essentially 0 after 30 layers.")

## 2. The same stack, same weights, with residual connections added

In [ ]:
x2 = x.detach().clone().requires_grad_(True)

h2 = x2
for layer in layers:  # exact same trained-nowhere weights as step 1
    h2 = h2 + torch.tanh(layer(h2))
loss2 = h2.sum()
loss2.backward()

print(f"Final layer activation std (forward signal): {h2.std().item():.2f}")
print(f"Gradient norm reaching the INPUT (backward signal): {x2.grad.norm().item():.2f}")

assert x2.grad.norm().item() > 1.0
assert x.grad.norm().item() < 1e-6
print("\nWith identical weights, the residual version keeps both signals alive across all 30 layers.")

## 3. Why: the gradient of `x + f(x)` always includes an identity term

By the sum rule, `d/dx [x + f(x)] = 1 + f'(x)`. No matter how small or
poorly-scaled `f'(x)` is, that `+1` guarantees at least some gradient
signal passes straight through every layer, unchanged. Stack 30 of these
and the gradient reaching the input is a product of terms each at least
`1 + (something)`, instead of a product of terms that can each be
arbitrarily small.

In [ ]:
# A minimal 1D version makes the identity term completely explicit
f_prime = torch.tensor(0.1)  # a deliberately small per-layer gradient contribution
n = 30

no_residual_factor = f_prime ** n
with_residual_factor = (1 + f_prime) ** n

print(f"Without residual, 30-layer gradient factor: {no_residual_factor:.2e}")
print(f"With residual,    30-layer gradient factor: {with_residual_factor:.2e}")
assert with_residual_factor > no_residual_factor * 1e6

## 4. Pre-norm vs. post-norm: where layer norm goes relative to the residual

The original Transformer paper normalized *after* adding the residual:
`x = LayerNorm(x + sublayer(x))` ("post-norm"). Modern architectures
(GPT-2 onward, and Module 16's transformer block here) normalize *before*
the sublayer instead: `x = x + sublayer(LayerNorm(x))` ("pre-norm") — this
keeps the identity path in the residual completely unobstructed by any
normalization, which empirically trains more stably at scale.

In [ ]:
class PostNormBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.sublayer = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(x + self.sublayer(x))


class PreNormBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.sublayer = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return x + self.sublayer(self.norm(x))


sample = torch.randn(2, d_model)
post = PostNormBlock(d_model)
pre = PreNormBlock(d_model)
print("post-norm output shape:", post(sample).shape)
print("pre-norm output shape: ", pre(sample).shape)
print("\nBoth are valid; Module 16 uses pre-norm, matching GPT-2/nanoGPT convention.")

## Recap

- Deep stacks without residual connections can suffer vanishing gradients
  and vanishing forward signal — verified concretely: gradient norm at the
  input dropped to ~1.5e-9 without residuals, vs. ~19 with them, using the
  exact same weights.
- The mechanism is simple: `d/dx[x + f(x)] = 1 + f'(x)` always has an
  identity term, guaranteeing at least some gradient flows through
  regardless of depth.
- Pre-norm (normalize, then apply the sublayer, then add the residual)
  keeps that identity path completely clean and is what modern transformer
  blocks use.

Modules 10-14 are now all the pieces a transformer block needs: attention,
positional encoding, layer norm, and residual connections. Module 15 adds
the last piece — the feed-forward block — before Module 16 assembles
everything into one real transformer block.